## Анализ метрик и когорт

In [24]:
import random
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

import os
from dotenv import load_dotenv

In [25]:
load_dotenv()

DB_USER = "postgres"
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "product_analytics"

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

np.random.seed(42)
random.seed(42)

engine = create_engine(DATABASE_URL)

### retention rate и churn rate

In [26]:
query = """
with session_dates as (
    select user_id, event_timestamp
    from events
    where event_name = 'session_start'
),
first_date as (
    select user_id, min(event_timestamp) as start_date
    from events
    where event_name = 'app_first_launch'
    group by user_id
),
metrics as (
    select 
        count(distinct case 
            when s.event_timestamp >= f.start_date + interval '24 hours'
             and s.event_timestamp <  f.start_date + interval '48 hours'
            then f.user_id 
        end)::float / nullif(count(distinct f.user_id), 0) as retention_rate_d1,

        count(distinct case 
            when s.event_timestamp >= f.start_date + interval '168 hours'
             and s.event_timestamp <  f.start_date + interval '192 hours'
            then f.user_id 
        end)::float / nullif(count(distinct f.user_id), 0) as retention_rate_d7,

        count(distinct case 
            when s.event_timestamp >= f.start_date + interval '720 hours'
             and s.event_timestamp <  f.start_date + interval '744 hours'
            then f.user_id 
        end)::float / nullif(count(distinct f.user_id), 0) as retention_rate_d30
    from first_date f
    left join session_dates s using(user_id)
)
select 
    retention_rate_d1,
    1 - retention_rate_d1 as churn_rate_d1,
    retention_rate_d7,
	1 - retention_rate_d7 as churn_rate_d7,
    retention_rate_d30,
	1 - retention_rate_d30 as churn_rate_d30
from metrics;
"""

df = pd.read_sql(query, engine)
df.head()

,retention_rate_d1,churn_rate_d1,retention_rate_d7,churn_rate_d7,retention_rate_d30,churn_rate_d30
0,0.171167,0.828833,0.113583,0.886417,0.013417,0.986583


### воронка пользователей

In [27]:
query = """
-- воронка
select
count(distinct(case when event_name = 'onboarding_complete' then user_id end))::float /
	nullif(count(distinct(case when event_name = 'app_first_launch' then user_id end)), 0) as conversion_rate_onboarding,
count(distinct(case when event_name = 'paywall_view' then user_id end))::float /
	nullif(count(distinct(case when event_name = 'onboarding_complete' then user_id end)), 0) as conversion_rate_paywall,
count(distinct(case when event_name = 'subscription_purchase' then user_id end))::float /
	nullif(count(distinct(case when event_name = 'paywall_view' then user_id end)), 0) as conversion_rate_purchase
from events
"""

df = pd.read_sql(query, engine)
df.head()

,conversion_rate_onboarding,conversion_rate_paywall,conversion_rate_purchase
0,0.801083,0.846978,0.116188


### ARPU

In [28]:
query = """
-- ARPU
select (SELECT sum(orders.amount) from orders where status = 'completed')::float 
	/ NULLIF((SELECT count(DISTINCT user_id) FROM users), 0) AS ARPU;
"""

df = pd.read_sql(query, engine)
df.head()

,arpu
0,1.534321


In [29]:
query = """
-- ARPPU
select sum(amount)::float / count(distinct(user_id)) as ARPPU
from orders
where status = 'completed';
"""

df = pd.read_sql(query, engine)
df.head()

,arppu
0,20.687472


In [30]:
month_number = '09'
month_name = 'september'

query = f"""
-- LTV_N по когортам
-- суммарная выручка от пользователей когорты за первые N дней с момента установки каждым пользователем
--		/ общее количество пользователей в когорте на day 0
-- не считаем за февраль, так как когорта "не созрела"

-- сентябрь

-- юзеры в нужном месяце
with needed_users as (
	select distinct(user_id) user_id, install_date
	from users
	where date_trunc('month', install_date) = '2025-{month_number}-01'
)

select 
(select count(*) from needed_users) as cohort_size,

sum(case when order_timestamp <= install_date + INTERVAL '7 days' then amount else 0 end) / (select count(*) from needed_users) as ltv_7_{month_name},
sum(case when order_timestamp <= install_date + INTERVAL '30 days' then amount else 0 end) / (select count(*) from needed_users) as ltv_30_{month_name}

from orders
inner join needed_users
	using(user_id)
where status = 'completed'
"""

df = pd.read_sql(query, engine)
df.head()

,cohort_size,ltv_7_september,ltv_30_september
0,623,1.050738,1.146854


In [31]:
query = """
-- Retention Matrix - процент пользователей, которые начавли пользоваться продуктом в определенную когорту 
-- (в данном случае - неделю) и продолжает возвращаться в продукт в последующие недели

-- только факт того, что юзер зашел
with user_activity as (
	select user_id, 
	date_trunc('week', install_date) as dt,
	max((event_timestamp <= install_date + interval '7 days')::int) as is_first_week_active,
	max((install_date + interval '7 days' < event_timestamp 
		and event_timestamp <= install_date + interval '14 days')::int) as is_second_week_active,
	max((install_date + interval '14 days' < event_timestamp 
		and event_timestamp <= install_date + interval '21 days')::int) as is_third_week_active,
	max((install_date + interval '21 days' < event_timestamp 
		and event_timestamp <= install_date + interval '28 days')::int) as is_fourth_week_active
	
	from users
	left join events
		using(user_id)
	group by user_id
)

-- к каждому пользователю прикручиваем была ли активность на следующих неделях
select dt, sum(is_first_week_active)::float / count(distinct(user_id)) as w0, sum(is_second_week_active)::float / count(distinct(user_id)) as w1, 
	sum(is_third_week_active)::float / count(distinct(user_id)) as w2, sum(is_fourth_week_active)::float / count(distinct(user_id)) as w3
from user_activity
group by dt
order by dt
"""

df = pd.read_sql(query, engine)
df.head(22)

,dt,w0,w1,w2,w3
0,2025-09-01,1.0,0.382353,0.205882,0.205882
1,2025-09-08,1.0,0.363636,0.281818,0.209091
2,2025-09-15,1.0,0.386503,0.257669,0.122699
3,2025-09-22,1.0,0.430962,0.284519,0.163180
4,2025-09-29,1.0,0.441379,0.268966,0.220690
5,2025-10-06,1.0,0.398810,0.258929,0.139881
6,2025-10-13,1.0,0.429319,0.253927,0.164921
7,2025-10-20,1.0,0.418093,0.259169,0.173594
8,2025-10-27,1.0,0.395699,0.283871,0.146237
9,2025-11-03,1.0,0.444219,0.286004,0.202840
